# Neural CFR+ Streaming Traversal Validation

This notebook validates the opt-in streamed/chunked GPU traversal path for neural CFR+. The old breadth-first GPU traversal remains the default; set `traversal_streaming=True` to use the new path.

The important checks are:

1. deterministic same-sample equivalence between old and streamed traversal on a small spec,
2. deterministic equivalence between unchunked streaming and aggressively chunked streaming,
3. a small committed-training smoke test,
4. a profiling cell for larger specs showing peak rows, chunks, memory, and record counts.


In [ ]:
from __future__ import annotations

import sys
import time
import types
from pathlib import Path

import numpy as np
import pandas as pd
import torch

REPO_ROOT = Path.cwd().resolve()
while not (REPO_ROOT / 'liars_poker').is_dir():
    if REPO_ROOT.parent == REPO_ROOT:
        raise RuntimeError('Could not find repo root containing liars_poker/')
    REPO_ROOT = REPO_ROOT.parent
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

from liars_poker.algo.deep_cfr_plus import DeepCFRPlusTrainer
from liars_poker.algo.neural_cfr_plus_gpu import GPUDeepCFRPlusTraverser
from liars_poker.core import GameSpec

assert torch.cuda.is_available(), 'Streaming traversal validation is intended for a CUDA machine.'
device = torch.device('cuda')
torch.set_float32_matmul_precision('high')

print('repo:', REPO_ROOT)
print('torch:', torch.__version__)
print('gpu:', torch.cuda.get_device_name(0))
free, total = torch.cuda.mem_get_info()
print('free / total GPU GiB:', round(free / 2**30, 2), '/', round(total / 2**30, 2))


In [ ]:
small_spec = GameSpec(
    ranks=4,
    suits=4,
    hand_size=2,
    claim_kinds=('RankHigh', 'Pair', 'TwoPair', 'Trips'),
    suit_symmetry=True,
)

large_profile_spec = GameSpec(
    ranks=6,
    suits=4,
    hand_size=4,
    claim_kinds=('RankHigh', 'Pair', 'TwoPair', 'Trips', 'FullHouse', 'Quads'),
    suit_symmetry=True,
)

def make_trainer(
    spec: GameSpec,
    *,
    streaming: bool,
    seed: int = 7,
    sample_count: int | None = None,
    sample_schedule: tuple[int, ...] | None = None,
    sample_mode: str = 'hash',
    sample_baseline: str = 'none',
    live_row_budget: int | None = None,
    action_chunk_size: int | None = None,
    traversal_batch_size: int = 64,
    buffer_capacity: int = 50_000,
) -> DeepCFRPlusTrainer:
    return DeepCFRPlusTrainer(
        spec,
        regret_hidden_sizes=(64, 64),
        strategy_hidden_sizes=(64, 64),
        device=device,
        seed=seed,
        regret_buffer_capacity=buffer_capacity,
        strategy_buffer_capacity=buffer_capacity,
        learning_rate=1e-3,
        batch_size=256,
        regret_train_steps=2,
        strategy_train_steps=2,
        regret_positive_weight=0.5,
        validation_fraction=0.0,
        traversal_backend='gpu_native',
        traversal_batch_size=traversal_batch_size,
        traverser_action_sample_count=sample_count,
        traverser_action_sample_schedule=sample_schedule,
        traverser_action_sample_mode=sample_mode,
        traverser_action_baseline=sample_baseline,
        traversal_streaming=streaming,
        traversal_live_row_budget=live_row_budget,
        traverser_action_chunk_size=action_chunk_size,
        traversal_record_flush_size=4096,
        device_replay=True,
        fused_optimizer=True,
        amp_dtype=None,
        compile_models=False,
    )

def install_deterministic_policy(traverser: GPUDeepCFRPlusTraverser) -> None:
    # Validation only: remove opponent multinomial-order differences by making
    # every strategy one-hot. Each node chooses the first legal claim when one
    # exists; otherwise it calls. Regret values remain deterministic and legal.
    def deterministic(self, actor, features, legal_mask):
        n, action_dim = legal_mask.shape
        base = torch.arange(action_dim, device=legal_mask.device, dtype=torch.float32)
        values = base.unsqueeze(0).expand(n, action_dim).clone()
        values = values + 0.001 * features[:, :1]
        claim_legal = legal_mask[:, 1:]
        has_claim = claim_legal.any(dim=1)
        first_claim = claim_legal.float().argmax(dim=1) + 1
        chosen = torch.where(has_claim, first_claim, torch.zeros_like(first_claim))
        strategy = torch.zeros_like(values)
        strategy.scatter_(1, chosen[:, None], 1.0)
        return values, strategy
    traverser._regrets_and_strategy = types.MethodType(deterministic, traverser)

def profile_once(
    trainer: DeepCFRPlusTrainer,
    *,
    traverser_id: int,
    batch_size: int,
    seed: int,
    deterministic_policy: bool = True,
) -> dict:
    trainer._gpu_traverser = GPUDeepCFRPlusTraverser(trainer)
    if deterministic_policy:
        install_deterministic_policy(trainer._gpu_traverser)
    torch.manual_seed(seed)
    torch.cuda.reset_peak_memory_stats(device)
    start = time.perf_counter()
    stats = trainer._gpu_traverser.run_traversals(
        traverser_id,
        batch_size,
        profile=True,
        commit_records=False,
    )
    torch.cuda.synchronize(device)
    stats['wall_s'] = time.perf_counter() - start
    return stats

def compare_stats(label: str, a: dict, b: dict, *, atol: float = 1e-5) -> dict:
    av = a['root_values']
    bv = b['root_values']
    max_abs = float((av - bv).abs().max().item()) if av.numel() else 0.0
    row = {
        'comparison': label,
        'root max abs diff': max_abs,
        'old regret records': a['regret_records'],
        'new regret records': b['regret_records'],
        'old strategy records': a['strategy_records'],
        'new strategy records': b['strategy_records'],
        'old sampled edges': a['sampled_claim_edges'],
        'new sampled edges': b['sampled_claim_edges'],
        'old peak rows': a.get('peak_rows'),
        'new peak rows': b.get('peak_rows'),
        'new edge chunks': b.get('streamed_edge_chunks', 0),
        'new row splits': b.get('streamed_row_splits', 0),
    }
    assert max_abs <= atol, row
    for key in ('regret_records', 'strategy_records', 'full_claim_edges', 'sampled_claim_edges'):
        assert a[key] == b[key], (label, key, a[key], b[key])
    return row


In [ ]:
# 1. Same-sample equivalence: full expansion, old breadth-first versus streamed.
old_full = make_trainer(small_spec, streaming=False, seed=11, sample_count=None)
stream_full = make_trainer(
    small_spec,
    streaming=True,
    seed=11,
    sample_count=None,
    live_row_budget=64,
    action_chunk_size=32,
)

old_stats = profile_once(old_full, traverser_id=0, batch_size=32, seed=123)
stream_stats = profile_once(stream_full, traverser_id=0, batch_size=32, seed=123)
pd.DataFrame([compare_stats('full expansion: old vs streamed', old_stats, stream_stats)])


In [ ]:
# 2. Same-sample equivalence: sampled traverser actions using deterministic hash sampling.
old_sampled = make_trainer(
    small_spec,
    streaming=False,
    seed=12,
    sample_count=4,
    sample_mode='hash',
    sample_baseline='call',
)
stream_sampled = make_trainer(
    small_spec,
    streaming=True,
    seed=12,
    sample_count=4,
    sample_mode='hash',
    sample_baseline='call',
    live_row_budget=48,
    action_chunk_size=24,
)

old_stats = profile_once(old_sampled, traverser_id=1, batch_size=40, seed=456)
stream_stats = profile_once(stream_sampled, traverser_id=1, batch_size=40, seed=456)
pd.DataFrame([compare_stats('sampled hash: old vs streamed', old_stats, stream_stats)])


In [ ]:
# 3. Chunking equivalence: large streamed chunks versus deliberately tiny chunks.
stream_big = make_trainer(
    small_spec,
    streaming=True,
    seed=13,
    sample_count=6,
    sample_mode='hash',
    sample_baseline='call',
    live_row_budget=None,
    action_chunk_size=None,
)
stream_tiny = make_trainer(
    small_spec,
    streaming=True,
    seed=13,
    sample_count=6,
    sample_mode='hash',
    sample_baseline='call',
    live_row_budget=16,
    action_chunk_size=8,
)

big_stats = profile_once(stream_big, traverser_id=0, batch_size=48, seed=789)
tiny_stats = profile_once(stream_tiny, traverser_id=0, batch_size=48, seed=789)
pd.DataFrame([compare_stats('streamed unchunked vs tiny chunks', big_stats, tiny_stats)])


In [ ]:
# 4. Committed-training smoke test. This is intentionally tiny; it checks that
# flushed records train without shape/device issues.
smoke = make_trainer(
    small_spec,
    streaming=True,
    seed=21,
    sample_count=4,
    sample_mode='hash',
    sample_baseline='call',
    live_row_budget=128,
    action_chunk_size=64,
    traversal_batch_size=32,
    buffer_capacity=20_000,
)

rows = []
for _ in range(3):
    rec = smoke.run_iteration(traversals_per_player=64)
    rows.append({
        'iteration': rec['iteration'],
        'regret records': sum(rec['new_regret_records']),
        'strategy records': sum(rec['new_strategy_records']),
        'sampled edges': rec['action_sampling']['sampled_claim_edges'],
        'edge fraction': rec['action_sampling']['claim_edge_fraction'],
        'streamed chunks': rec['action_sampling'].get('streamed_edge_chunks', 0),
        'row splits': rec['action_sampling'].get('streamed_row_splits', 0),
        'traversal s': rec['timing']['traversal_s'],
        'regret fit s': rec['timing']['regret_training_s'],
        'strategy fit s': rec['timing']['strategy_training_s'],
    })

pd.DataFrame(rows)


In [ ]:
# 5. Larger-spec profiling: old versus streamed on the 69-claim spec.
# Increase batch sizes here on the GPU machine after the small checks pass.
profile_rows = []
profile_configs = [
    ('old breadth-first cap16', False, 16, None, None, 64),
    ('streamed cap16 row budget 4096', True, 16, 4096, 8192, 256),
    ('streamed full first two then cap16', True, None, 4096, 8192, 128),
]

for label, streaming, sample_count, live_budget, chunk_size, batch_size in profile_configs:
    schedule = (69, 69, 16) if label.startswith('streamed full') else None
    trainer = make_trainer(
        large_profile_spec,
        streaming=streaming,
        seed=31,
        sample_count=sample_count,
        sample_schedule=schedule,
        sample_mode='hash',
        sample_baseline='call',
        live_row_budget=live_budget,
        action_chunk_size=chunk_size,
        traversal_batch_size=batch_size,
        buffer_capacity=10_000,
    )
    stats = profile_once(
        trainer,
        traverser_id=0,
        batch_size=batch_size,
        seed=999,
        deterministic_policy=False,
    )
    profile_rows.append({
        'configuration': label,
        'batch size': batch_size,
        'wall s': stats['wall_s'],
        'regret records': stats['regret_records'],
        'strategy records': stats['strategy_records'],
        'full edges': stats['full_claim_edges'],
        'sampled edges': stats['sampled_claim_edges'],
        'peak rows': stats.get('peak_rows'),
        'streamed chunks': stats.get('streamed_edge_chunks', 0),
        'row splits': stats.get('streamed_row_splits', 0),
        'peak allocated GiB': stats.get('peak_allocated_bytes', 0) / 2**30,
        'peak reserved GiB': stats.get('peak_reserved_bytes', 0) / 2**30,
    })

pd.DataFrame(profile_rows).sort_values('configuration')
